# Silver schedules

## Pipeline overview

**Purpose:** Create a clean, source-shaped game schedule at one row per `game_id`. Keep future games and detailed playoff rounds. This notebook does not decide which games count in a Gold metric. Existing notebooks and Silver outputs remain unchanged.

| Role | Files | Current columns | Grain |
| --- | --- | ---: | --- |
| Primary input | `data/bronze/schedules/schedules_{season}.parquet` | 46 | `game_id` |
| Optional diagnostic input | `data/bronze/pbp/pbp_{season}.parquet` | 372 | `game_id + play_id` |
| Output | `data/silver/schedules/silver_schedules_{season}.parquet` | 46 | `game_id` |

**Seasons:** 2023–2026. The active 2026 schedule currently has regular-season games only; neither 2026 row count nor postseason coverage is hard-coded. PBP is used only to report games with plays but no schedule score.

### Cleaning and transformations

1. Preserve all source columns and column order. Remove only exact full-row duplicates; stop on conflicting records for one `game_id`.
2. Parse `gameday` as a datetime. Validate `gametime` as `HH:MM` but keep the source string because its timezone is not established here.
3. Convert whole-number provider IDs, scores, result, total, odds, and binary indicators to nullable integers. Keep text IDs as strings and fractional betting lines as floats. Preserve future-game nulls.

### Validation and quality checks

- Required files, 46 named columns, safe integer/date/time conversion, exact-duplicate counts, and row reconciliation.
- Non-null, unique `game_id` within and across seasons; season/week/game type/date consistency; distinct home and away teams; no team scheduled twice in one NFL week.
- On scored games, nonnegative scores and exact `result = home_score - away_score` and `total = home_score + away_score`; null scores on future games remain null.
- Source-value reconciliation except approved type changes and exact-row deduplication; 46-column Parquet round trip.
- PBP-versus-score coverage is diagnostic, not a hard failure: a game can have plays before schedule results refresh.


## 1. Set scope and type contract

Provider IDs such as `gsis`, `pff`, and `ftn` are numeric in Bronze and remain numeric identifiers. `game_id`, `old_game_id`, `espn`, and other textual IDs are not recoded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


SEASONS = [2023, 2024, 2025, 2026]
EXPECTED_SCHEDULE_COLUMNS = 46
GAME_TYPES = {"REG", "WC", "DIV", "CON", "SB"}

INTEGER_COLUMNS = [
    "gsis", "pff", "ftn",
    "away_score", "home_score", "result", "total",
    "away_rest", "home_rest",
    "away_moneyline", "home_moneyline",
    "away_spread_odds", "home_spread_odds",
    "under_odds", "over_odds",
]
BINARY_COLUMNS = ["overtime", "div_game"]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    PROJECT_ROOT = next(
        parent for parent in PROJECT_ROOT.parents
        if (parent / "ROADMAP.md").exists()
    )

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_SCHEDULES_DIR = PROJECT_ROOT / "data" / "silver" / "schedules"
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/dtwice/Development/dev_sports/nfl/fantasy-football/nfl-dream-lab-app


## 2. Inventory source schemas

The source has 46 named columns in each current season. Nullable game-result fields have different Parquet physical types during an active season, so names and order are the schema-shape contract; selected fields are normalized below.

In [2]:
source_columns = None
source_inventory = []

for season in SEASONS:
    schedule_path = BRONZE_DIR / "schedules" / f"schedules_{season}.parquet"
    assert schedule_path.exists(), f"Missing schedule: {schedule_path}"
    source_schema = pq.read_schema(schedule_path)
    assert len(source_schema) == EXPECTED_SCHEDULE_COLUMNS, (season, len(source_schema))
    assert set(INTEGER_COLUMNS + BINARY_COLUMNS + [
        "game_id", "season", "week", "game_type", "gameday",
        "gametime", "weekday", "home_team", "away_team",
    ]).issubset(source_schema.names)
    if source_columns is None:
        source_columns = source_schema.names
    else:
        assert source_schema.names == source_columns, f"Column drift in {season}"
    source_inventory.append({
        "season": season,
        "source_rows": pq.ParquetFile(schedule_path).metadata.num_rows,
        "source_columns": len(source_schema),
    })

pd.DataFrame(source_inventory)

,season,source_rows,source_columns
0,2023,285,46
1,2024,285,46
2,2025,285,46
3,2026,272,46


## 3. Load and deduplicate

Exact duplicates are removed before normalization. A repeated game key with different values fails instead of choosing a winner.

In [3]:
bronze_schedules = {}
deduplicated_schedules = {}
duplicate_counts = {}

for season in SEASONS:
    source_path = BRONZE_DIR / "schedules" / f"schedules_{season}.parquet"
    bronze = pd.read_parquet(source_path)
    assert bronze.columns.tolist() == source_columns
    assert bronze["season"].eq(season).all()

    duplicate_mask = bronze.duplicated(keep="first")
    deduplicated = bronze.loc[~duplicate_mask].reset_index(drop=True)
    assert deduplicated["game_id"].notna().all()
    assert deduplicated["game_id"].map(type).eq(str).all()
    assert not deduplicated.duplicated("game_id").any(), (season, "conflicting game keys")

    bronze_schedules[season] = bronze
    deduplicated_schedules[season] = deduplicated
    duplicate_counts[season] = int(duplicate_mask.sum())

pd.DataFrame({
    "season": SEASONS,
    "bronze_rows": [len(bronze_schedules[season]) for season in SEASONS],
    "exact_duplicates_removed": [duplicate_counts[season] for season in SEASONS],
    "silver_rows_expected": [len(deduplicated_schedules[season]) for season in SEASONS],
})

,season,bronze_rows,exact_duplicates_removed,silver_rows_expected
0,2023,285,0,285
1,2024,285,0,285
2,2025,285,0,285
3,2026,272,0,272


## 4. Normalize selected types and reconcile values

Integer conversion rejects fractional, non-finite, and unsafe-magnitude values. Missing results for future games stay missing. `gametime` remains the original text after syntax validation.

In [4]:
silver_schedules = {}
changed_columns = set(INTEGER_COLUMNS + BINARY_COLUMNS + ["gameday"])
untouched_columns = [name for name in source_columns if name not in changed_columns]

for season in SEASONS:
    source = deduplicated_schedules[season]
    silver = source.copy()

    for column in INTEGER_COLUMNS + BINARY_COLUMNS:
        numeric_values = pd.to_numeric(silver[column], errors="raise")
        populated = numeric_values.dropna()
        assert np.isfinite(populated).all(), (season, column, "non-finite")
        assert populated.mod(1).eq(0).all(), (season, column, "fractional")
        assert populated.abs().le(2**53).all(), (season, column, "unsafe float integer")
        if column in BINARY_COLUMNS:
            assert populated.isin([0, 1]).all(), (season, column, "not 0/1")
            silver[column] = numeric_values.astype("Int8")
        else:
            silver[column] = numeric_values.astype("Int64")

    silver["gameday"] = pd.to_datetime(
        silver["gameday"], format="%Y-%m-%d", errors="raise"
    )
    assert silver["gameday"].notna().all()
    assert pd.to_datetime(silver["gametime"], format="%H:%M", errors="coerce").notna().all()

    pd.testing.assert_frame_equal(source[untouched_columns], silver[untouched_columns])
    for column in INTEGER_COLUMNS + BINARY_COLUMNS:
        pd.testing.assert_series_equal(
            pd.to_numeric(source[column]).astype("Float64"),
            silver[column].astype("Float64"),
            check_names=False,
        )
    assert silver["gameday"].dt.strftime("%Y-%m-%d").equals(source["gameday"])
    assert silver.columns.tolist() == source_columns
    assert len(silver) == len(bronze_schedules[season]) - duplicate_counts[season]
    silver_schedules[season] = silver

print("Selected types normalized and all source values reconciled.")

Selected types normalized and all source values reconciled.


## 5. Validate schedule meaning

NFL season is not necessarily the calendar year of a postseason game. Future games may lack scores, odds, weather, or result fields; those nulls are not errors.

In [5]:
all_game_ids = pd.concat(
    [silver_schedules[season]["game_id"] for season in SEASONS], ignore_index=True
)
assert not all_game_ids.duplicated().any(), "game_id repeated across seasons"

for season in SEASONS:
    schedules = silver_schedules[season]
    assert schedules["game_id"].notna().all()
    assert not schedules.duplicated("game_id").any()
    assert schedules["season"].eq(season).all()
    assert schedules["week"].between(1, 22).all()
    assert schedules["game_type"].isin(GAME_TYPES).all()
    assert schedules["gameday"].dt.year.isin([season, season + 1]).all()
    assert schedules["weekday"].eq(schedules["gameday"].dt.day_name()).all()
    assert schedules[["home_team", "away_team"]].notna().all().all()
    assert schedules["home_team"].ne(schedules["away_team"]).all()

    home_weeks = schedules[["season", "week", "home_team"]].rename(
        columns={"home_team": "team"}
    )
    away_weeks = schedules[["season", "week", "away_team"]].rename(
        columns={"away_team": "team"}
    )
    team_weeks = pd.concat([home_weeks, away_weeks], ignore_index=True)
    assert not team_weeks.duplicated(["season", "week", "team"]).any()

    scored = schedules["home_score"].notna() & schedules["away_score"].notna()
    assert schedules["home_score"].isna().eq(schedules["away_score"].isna()).all()
    assert schedules.loc[scored, ["result", "total"]].notna().all().all()
    assert schedules.loc[scored, ["home_score", "away_score"]].ge(0).all().all()
    assert schedules.loc[scored, "result"].eq(
        schedules.loc[scored, "home_score"] - schedules.loc[scored, "away_score"]
    ).all()
    assert schedules.loc[scored, "total"].eq(
        schedules.loc[scored, "home_score"] + schedules.loc[scored, "away_score"]
    ).all()
    assert schedules[["home_rest", "away_rest"]].ge(0).all().all()

print("Game keys, calendar context, teams, scores, and totals validated.")

Game keys, calendar context, teams, scores, and totals validated.


## 6. Diagnose PBP-versus-score coverage

This does not gate Silver publication. PBP and schedule scores can refresh at different times; neither source alone establishes a universally safe `is_completed` definition for future Gold metrics.

In [6]:
pbp_score_diagnostics = []

for season in SEASONS:
    pbp_path = BRONZE_DIR / "pbp" / f"pbp_{season}.parquet"
    if not pbp_path.exists():
        pbp_score_diagnostics.append({"season": season, "pbp_games": pd.NA, "pbp_games_without_schedule_score": pd.NA, "game_ids": []})
        continue

    pbp_game_ids = set(pd.read_parquet(pbp_path, columns=["game_id"])["game_id"].dropna())
    schedules = silver_schedules[season]
    scored_game_ids = set(schedules.loc[
        schedules["home_score"].notna() & schedules["away_score"].notna(), "game_id"
    ])
    missing_score_game_ids = sorted(pbp_game_ids - scored_game_ids)
    pbp_score_diagnostics.append({
        "season": season,
        "pbp_games": len(pbp_game_ids),
        "pbp_games_without_schedule_score": len(missing_score_game_ids),
        "game_ids": missing_score_game_ids,
    })

pd.DataFrame(pbp_score_diagnostics)

,season,pbp_games,pbp_games_without_schedule_score,game_ids
0,2023,285,0,[]
1,2024,285,0,[]
2,2025,285,0,[]
3,2026,80,1,[2026_06_SEA_DEN]


## 7. Persist and verify each season

The saved files keep all 46 source fields. A fresh read must reproduce each in-memory Silver partition exactly.

In [7]:
SILVER_SCHEDULES_DIR.mkdir(parents=True, exist_ok=True)
validation_results = []

for season in SEASONS:
    schedules = silver_schedules[season]
    output_path = SILVER_SCHEDULES_DIR / f"silver_schedules_{season}.parquet"
    schedules.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, schedules)
    assert reloaded.columns.tolist() == source_columns
    assert len(reloaded.columns) == EXPECTED_SCHEDULE_COLUMNS
    assert pd.api.types.is_datetime64_any_dtype(reloaded["gameday"])
    assert all(str(reloaded[column].dtype) == "Int64" for column in INTEGER_COLUMNS)
    assert all(str(reloaded[column].dtype) == "Int8" for column in BINARY_COLUMNS)

    validation_results.append({
        "season": season,
        "bronze_rows": len(bronze_schedules[season]),
        "exact_duplicates_removed": duplicate_counts[season],
        "silver_rows": len(reloaded),
        "columns": len(reloaded.columns),
        "scored_games": int(reloaded[["home_score", "away_score"]].notna().all(axis=1).sum()),
        "output": str(output_path.relative_to(PROJECT_ROOT)),
    })
    print(f"Validated and wrote {output_path.name}")

Validated and wrote silver_schedules_2023.parquet
Validated and wrote silver_schedules_2024.parquet
Validated and wrote silver_schedules_2025.parquet
Validated and wrote silver_schedules_2026.parquet


## 8. Review the output summary

All four partitions must pass validation. The active-season score count can change on a later Bronze refresh; it is not used to filter the Silver schedule.

In [8]:
validation_summary = pd.DataFrame(validation_results)
assert len(validation_summary) == len(SEASONS)
assert validation_summary["silver_rows"].eq(
    validation_summary["bronze_rows"] - validation_summary["exact_duplicates_removed"]
).all()
validation_summary

,season,bronze_rows,exact_duplicates_removed,silver_rows,columns,scored_games,output
0,2023,285,0,285,46,285,data/silver/schedules/silver_schedules_2023.pa...
1,2024,285,0,285,46,285,data/silver/schedules/silver_schedules_2024.pa...
2,2025,285,0,285,46,285,data/silver/schedules/silver_schedules_2025.pa...
3,2026,272,0,272,46,79,data/silver/schedules/silver_schedules_2026.pa...
